In [1]:
import pandas as pd
from tqdm import tqdm
from dataset import MyriadLamaDataset

dataset = MyriadLamaDataset(model_name="llama3.2_3b_it")
dataloader = dataset.get_dataloader(batch_size=8, shuffle=False)


Dataset already exists at /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/llama3.2_3b_it/paraphrases_dataset. Loading from disk.


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/net/tokyo100-10g/data/str01_01/xzhao/models/llama_hf/llama3.2_3b_it"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path, device_map="cuda:0", torch_dtype="auto"
)
few_shot_examples = dataset.get_few_shot_examples(k=1)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
sample_count = 0
for uuids, answers, all_paraphrases in tqdm(dataloader):
    batch_predictions = []
    batch_generations = []
    batch_templates = []

    # Process each question in batch
    for i, paraphrases in enumerate(zip(*all_paraphrases)):
        # All paraphrases in MyriadLAMA are manually generated
        # Simply select the first N paraphrases
        all_templates = list(paraphrases)
        selected_templates = all_templates[: 2]

        prompt, segment_metadata = dataset.construct_prompts_with_paraphrases(
            few_shot_examples, paraphrases=selected_templates
        )
        
        break
    break


  0%|          | 0/250 [00:00<?, ?it/s]


In [4]:
# generation = myriadlama_flex_generation(prompt, segment_metadata, max_new_tokens=10)
import torch
from transformers import BatchEncoding
from generate_myriadlama2 import FlexAttentionWrapper
from generate_myriadlama2 import tokenize_with_segment
from generate_myriadlama2 import create_myriadlama_mask_mod

tokenizer.pad_token_id = tokenizer.eos_token_id
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.pad_token_id = tokenizer.eos_token_id

paraphrases = []
for i, length in enumerate(segment_metadata["len_paras"]):
    start = segment_metadata["len_context"] + sum(segment_metadata["len_paras"][:i])
    end = start + length
    paraphrases.append(prompt[start:end])

# Process prompt with position tracking and metadata
concatenated_text, full_tokens, segment_positions, original_length = (
    tokenize_with_segment(prompt, tokenizer, segment_metadata)
)



✅ FlexAttention is available


In [ ]:
position_ids = torch.arange(len(full_tokens), dtype=torch.long, device=model.device)
context_end = segment_positions[0]['end']

start_generation_token_id = context_end + max(segment['end'] - segment['start'] for segment in segment_positions[1:])

for segment in segment_positions[1:]:
    position_ids[segment['start']:segment['end']] = torch.arange(
        0, segment['end'] - segment['start'], dtype=torch.long, device=model.device
    ) + position_ids[context_end - 1] + 1

inputs = {
    "input_ids": torch.tensor([full_tokens]), 
    "attention_mask": torch.ones(1, len(full_tokens))
}
inputs = BatchEncoding(data=inputs).to(model.device)

position_ids = position_ids.unsqueeze(0).expand_as(inputs["input_ids"]) 

[{'start': 0, 'end': 36, 'type': 'context'}, {'start': 36, 'end': 51, 'type': 'paraphrase'}, {'start': 51, 'end': 69, 'type': 'paraphrase'}]


In [17]:
position_ids, start_generation_token_id

(tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
          18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
          36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 36, 37, 38,
          39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]],
        device='cuda:0'),
 53)

In [ ]:


flex_wrapper = FlexAttentionWrapper(model)

print(concatenated_text)

generated = None
mask_mod = create_myriadlama_mask_mod(
    segment_positions, segment_metadata, original_length
)

# Generation loop
for step in range(10):
    flex_wrapper.patch_model(mask_mod)
    logits = model(inputs["input_ids"]).logits[:, -1, :]
    break

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# --- 2. Create Dummy Data ---
S = 15 # Total sequence length
# Create a grid of indices
q_idx = torch.arange(S).unsqueeze(1) # Column
kv_idx = torch.arange(S).unsqueeze(0) # Row

# Define Segments: Context=[0-5), Para1=[5-10), Para2=[10-15)
# Indices 15+ are "generated" tokens
seg_starts = torch.tensor([0, 5, 10])
seg_ends = torch.tensor([5, 10, 15])
original_length = 15

# --- 3. Run Logic ---
causal_mask = q_idx >= kv_idx
is_generated = q_idx >= original_length

s_starts = seg_starts[:, None, None]
s_ends = seg_ends[:, None, None]

q_grid_exp = q_idx.unsqueeze(0)
kv_grid_exp = kv_idx.unsqueeze(0)

# Segments
# Note: We simulate the 'b' and 'h' dimensions by broadcasting or ignoring them
q_in_segment = (q_grid_exp >= s_starts) & (q_grid_exp < s_ends)
kv_in_segment = (kv_grid_exp >= s_starts) & (kv_grid_exp < s_ends)

# Get IDs (collapse segment dimension)
q_seg_id = q_in_segment.to(torch.int32).argmax(dim=0)
kv_seg_id = kv_in_segment.to(torch.int32).argmax(dim=0)

# Logic Rules
kv_is_context = (kv_seg_id == 0)
same_segment = (q_seg_id == kv_seg_id)

# Intra-segment OR Context
intra_segment_or_context = same_segment | kv_is_context

valid_topology = torch.where(
    is_generated, 
    torch.tensor(True), 
    intra_segment_or_context
)
mask = valid_topology & causal_mask

# --- 4. Visualize ---
plt.figure(figsize=(8, 6))
# Cast to float for heatmap (0.0 vs 1.0)
sns.heatmap(mask.float(), cmap="Greys_r", cbar=False, square=True, linewidths=0.5, linecolor='gray')

# Add Labels for clarity
plt.title("White = Attended, Black = Masked")
plt.xlabel("Keys (kv_idx)")
plt.ylabel("Queries (q_idx)")

In [ ]:
q_seg_id.shape, kv_seg_id.shape, kv_is_context.shape

In [ ]:
same_segment.int()

In [ ]:
from generate_myriadlama2 import tokenize_with_segment


context = prompt[: segment_metadata["len_context"]]
paraphrases = []
for i, length in enumerate(segment_metadata["len_paras"]):
    start = segment_metadata["len_context"] + sum(segment_metadata["len_paras"][:i])
    end = start + length
    paraphrases.append(prompt[start:end])

concatenated_text, full_tokens, segment_positions, original_length = (
    tokenize_with_segment(prompt, tokenizer, segment_metadata)
)

In [ ]:
tokenizer(context, return_tensors="pt", truncation=True, add_special_tokens=True)

In [ ]:
for item in positions:
    print(item["type"])
    print(tokenizer.decode(all_tokens[item["start"]: item["end"]]))

In [ ]:
from generate_myriadlama import construct_prompt_new_format


for uuids, answers, all_paraphrases in tqdm(dataloader):
    batch_predictions = []
    batch_generations = []
    batch_templates = []

    # Process each question in batch
    for i, paraphrases in enumerate(zip(*all_paraphrases)):
        # All paraphrases in MyriadLAMA are manually generated
        # Simply select the first N paraphrases
        all_templates = list(paraphrases)
        selected_templates = all_templates[: 5]

        # Construct ONE prompt with ALL question paraphrases (NEW FORMAT)
        # Prompt has: instruction + few-shot examples + ALL main question paraphrases
        prompt = construct_prompt_new_format(
            dataset.instruction,
            few_shot_examples,
            selected_templates,  # Pass ALL paraphrases, not just one
        )
        break
    break

In [ ]:
from generate_myriadlama import concatenate_paraphrases_with_positions
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/net/tokyo100-10g/data/str01_01/xzhao/models/llama_hf/llama3.2_3b_it"
tokenizer = AutoTokenizer.from_pretrained(model_path)
concatenated_text, segment_positions, segment_metadata, original_length = (
    concatenate_paraphrases_with_positions(prompt, tokenizer)
)

from generate_myriadlama import create_myriadlama_mask

mask_mod = create_myriadlama_mask(
        segment_positions, segment_metadata, original_length
)

from generate_myriadlama import FlexAttentionWrapper


model = AutoModelForCausalLM.from_pretrained(
    model_path, device_map="cuda:0", torch_dtype="auto"
)
flex_wrapper = FlexAttentionWrapper(model)

flex_wrapper.patch_model(mask_mod)

In [ ]:
inputs = tokenizer(
    concatenated_text, return_tensors="pt", truncation=True, add_special_tokens=True
).to(model.device)


In [ ]:
logits = model(inputs["input_ids"]).logits[:, -1, :]

In [ ]:
logits